<a href="https://colab.research.google.com/github/LukacsHunor/BEST/blob/main/FRBS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras.initializers import RandomUniform
from keras.layers import Layer

@tf.function
def trapmf(x, abcd):
    a, b, c, d = abcd[0], abcd[1], abcd[2], abcd[3]

    assertion = tf.reduce_all(tf.logical_and(a <= b, tf.logical_and(b <= c, c <= d)))
    tf.debugging.assert_equal(assertion, True, message=f'Ensure that a <= b <= c <= d! The values are: a={a}, b={b}, c={c}, d={d}')

    y = tf.zeros_like(x, dtype=tf.float32)  # Initialize output tensor

    # Calculate membership values based on the trapezoidal membership function
    y = tf.where(tf.logical_and(a < x, x < b), (x - a) / (b - a), y)
    y = tf.where(tf.logical_and(b <= x, x <= c), 1.0, y)
    y = tf.where(tf.logical_and(c < x, x < d), (d - x) / (d - c), y)

    return y

@tf.function
def antes(rule):
    return rule[:-1]

@tf.function
def w_min(rule, observations):

    obs = tf.transpose(observations)
    tmp = tf.map_fn(lambda x: trapmf(x[1], x[0]), (antes(rule), obs), dtype=tf.float32)
    return tf.reduce_min(tmp, axis=0)

@tf.function
def mamdaniInference(frbs: tf.Tensor, observations: tf.Tensor) -> tf.Tensor:
    _observation_len = tf.shape(observations)[0]

    Ys = tf.zeros(_observation_len)

    w_mins = tf.map_fn(lambda rule: w_min(rule=rule, observations=observations), frbs)
    w_mins = tf.cast(w_mins, tf.float32)
    consequents_mtx = frbs[:, -1]

    A = tf.expand_dims(consequents_mtx[:, 0], axis=-1)
    B = tf.expand_dims(consequents_mtx[:, 1], axis=-1)
    C = tf.expand_dims(consequents_mtx[:, 2], axis=-1)
    D = tf.expand_dims(consequents_mtx[:, 3], axis=-1)

    _numerator_mtx = 3 * w_mins * (D**2 - A**2) * (1 - w_mins) + 3 * w_mins**2 * (C * D - A * B) + w_mins**3 * (C - D + A - B) * (C - D - A + B)
    _denominator_mtx = 2 * w_mins * (D - A) + w_mins**2 * (C + A - D - B) + tf.keras.backend.epsilon()

    _numerator = tf.reduce_sum(_numerator_mtx, axis=0)
    _denominator = tf.reduce_sum(_denominator_mtx, axis=0)

    Ys = tf.where(_denominator != 0, _numerator / _denominator, tf.zeros_like(_numerator))

    Ys *= (1 / 3)
    return Ys

@tf.function
def mamdaniInference_AntesCons(Antes: tf.Tensor, Cons: tf.Tensor, inputs: tf.Tensor) -> tf.Tensor:
    assert Antes.shape[1] == inputs.shape[1], 'The Antecedents input dimension must match with the input dimension (observation_dim)!'
    assert Antes.shape[0] == Cons.shape[0], 'The Antecedents n_rules must match with the Cons n_rules!'
    n_rules, in_dim, _ = Antes.shape
    zeros_tensor = tf.zeros((n_rules, 1, 4), dtype=Antes.dtype)
    frbs = tf.concat([Antes, zeros_tensor], axis=1)
    _output_dim = Cons.shape[1]

    Ys = []

    for cons_i in range(_output_dim):
        frbs = tf.concat([frbs[:, :-1], Cons[:, cons_i:cons_i+1]], axis=1)
        Ys.append(mamdaniInference(frbs=frbs, observations=inputs))

    return tf.stack(Ys, axis=1)



class BreakpointIncreasingOrderConstraint(tf.keras.constraints.Constraint):
    def __call__(self, w):
        '''
        >>> sort_blocks(tf.constant([1,3,2,4,8,7,6,5]))
        >>> # <tf.Tensor: shape=(8,), dtype=int32, numpy=array([1, 2, 3, 4, 5, 6, 7, 8], dtype=int32)>
        '''
        # Reshape the array to have 4 columns
        reshaped = tf.reshape(w, (-1, 4))
        # Sort each block along the last axis
        sorted_blocks = tf.sort(reshaped, axis=-1)
        # Concatenate the sorted blocks back together while maintaining the original shape
        sorted_array = tf.reshape(sorted_blocks, tf.shape(w))
        return sorted_array


class FuzzyLayer(Layer):
    def __init__(self, in_dim, out_dim, nr_rules):
        super().__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.nr_rules = nr_rules

        # Define the range for initialization
        min_val = -0.3
        max_val = 1.3

        self.Antes = self.add_weight(shape=(self.nr_rules, self.in_dim, 4),
                                      initializer=RandomUniform(min_val, max_val), # 'random_normal',
                                      trainable=True,
                                      dtype=tf.float32,
                                      name="Antecedents",
                                      constraint=BreakpointIncreasingOrderConstraint())
        self.Cons = self.add_weight(shape=(self.nr_rules, self.out_dim, 4),
                                     initializer=RandomUniform(min_val, max_val), #'random_normal',
                                     trainable=True,
                                     dtype=tf.float32,
                                     name="Consequents",
                                     constraint=BreakpointIncreasingOrderConstraint())

        self.Antes.assign(BreakpointIncreasingOrderConstraint()(self.Antes))
        self.Cons.assign(BreakpointIncreasingOrderConstraint()(self.Cons))

    # def build(self, input_shape):
    #     # Set the input shape and output shape
    #     self.input_shape = (self.in_dim,)
    #     self.output_shape = (self.out_dim,)

    # def compute_output_shape(self, input_shape):
    #     # This method should return a tuple that specifies the shape of the output
    #     # In your case, the output shape is (self.out_dim,)
    #     return (input_shape[0], self.out_dim)

    def call(self, inputs):
        return self.inference(inputs)

    def inference(self, inputs):
        return mamdaniInference_AntesCons(Antes=self.Antes, Cons=self.Cons, inputs=inputs)

    def get_trainable_params(self):
        trainable_weights = [tf.reshape(w, [-1]) for w in self.trainable_variables]
        trainable_weights_1d = tf.concat(trainable_weights, axis=0)
        return trainable_weights_1d.numpy()


    def summary(self, show_params=False):
        print("FuzzyLayer Summary:")
        print(f"{'Input Dimension:':<20} {self.in_dim}")
        print(f"{'Output Dimension:':<20} {self.out_dim}")
        print(f"{'Number of Rules:':<20} {self.nr_rules}")
        print(f"{'Antes shape:':<20} {self.Antes.shape}")
        print(f"{'Cons shape:':<20} {self.Cons.shape}")
        if show_params:
            print("Antes params:")
            print(self.Antes.numpy())
            print("Cons params:")
            print(self.Cons.numpy())





# @tf.function
# def mamdaniInference(frbs: tf.Tensor, observations: tf.Tensor) -> tf.Tensor:
#     """
#     Predict output values for the observations, using Mamdani Inference and Center of Gravity method.
#     We're using the explicit formula.
#     The length of the Outputs will be the same as the length of the Observations.

#     Parameters:
#     ----------
#     frbs: Fuzzy rule-based system
#     observations: Observations. Dimensionality of the observations must match with the number of Antecedents in a Rule.

#     Result:
#     -----
#     Ys: Output values for the observations according to the frbs

#     Example:
#     >>> frbs = np.array([[[1,2,2,3],[1,3,3,5],[3,4,4,5]], [[2,4,4,6],[1,2,2,3],[4,5.5,5.5,7]]])
#     >>> obs = np.array([[2.6,2.4]])
#     >>> mamdaniInference(frbs=frbs, observations=obs)
#     """



# @tf.function
# def mamdaniInference_AntesCons(Antes: tf.Tensor, Cons: tf.Tensor, inputs: tf.Tensor) -> tf.Tensor:
#     """
#     Predict output values for the observations, using Mamdani Inference and Center of Gravity method,
#     given the antecedents, consequents, and inputs.

#     Parameters:
#     ----------
#     Antes: Antecedents tensor (n_rules, in_dim, 4)
#     Cons: Consequents tensor (n_rules, out_dim, 4)
#     inputs: Observations tensor (n_samples, in_dim)

#     Returns:
#     -----
#     Ys: Output values for the observations according to the frbs (n_samples, out_dim)

#     Example:
#     >>> Antes = tf.constant([[[1,2,2,3],[1,3,3,5],[3,4,4,5]], [[2,4,4,6],[1,2,2,3],[4,5.5,5.5,7]]])
#     >>> Cons = tf.constant([[1, 2, 3], [2, 3, 4]])
#     >>> inputs = tf.constant([[2.6, 2.4]])
#     >>> result = mamdaniInference_AntesCons(Antes, Cons, inputs)
#     >>> print(result)
#     """

# @tf.function
# def w_min(rule, observations):
#     """
#     W_min calculator

#     Parameters
#     ----------
#     observations: tf.Tensor, shape: (number of inputs, dim of input)
#         Observations

#     Returns
#     -------
#     w: tf.Tensor, shape: (number of inputs,)
#         W_min for each input
#     """

# @tf.function
# def trapmf(x, abcd):
#     """
#     Trapezoidal membership function.
#     It returns the values where the crisp input intersects the trapeze.

#     Parameters
#     ----------
#     x: tf.Tensor
#         Independent variable
#     abcd: tf.Tensor
#         Breakpoints. Ensure that a <= b <= c <= d.

#     Returns
#     -------
#     y: tf.Tensor
#         Output values for the x values, according to the membership function

#     Example:
#     >>> trapmf(tf.constant([1.5]), [1, 2, 3, 5])
#     """


# @tf.function
# def antes(rule):
#     """
#     Returns the antecedents of the rule system.

#     Parameters
#     ----------
#     rule: tf.Tensor
#         Rule system

#     Returns
#     -------
#     tf.Tensor
#         Antecedents of the rule system
#     """

In [ ]:
print("asd")

asd


In [1]:
print("test")

test
